# How many bikes each plant assembles each month, with a central warehouse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/central_warehouse.ipynb)

A company assembles electric bicycles in three plants and plans, from January to June, how many bikes each plant assembles each month. The second version of the model, M1, keeps bikes in stock at each plant. With a third demand forecast, which has a peak in February, M1 has no solution: the plants cannot assemble enough bikes in advance. The orders can still be served if some of them are served one month late, and M1 does not represent that. This notebook uses the third version, M2. The plants ship to a central warehouse, which holds stock and serves the orders, some of them late. The plants and the warehouse can hold only a limited number of bikes. The notebook follows the plan in steps:

1. The data of the plants, the warehouse and the demand forecast.
2. The model with the central warehouse.
3. The plan with forecast 3.
4. Four tests, each one changing a single datum: the limit on the orders owed at the end of June, the storage capacity of the plants, the cost of serving late and the storage capacity of the warehouse.

**How to use it.** Run the cells in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and tick boxes): change the values and run the cell again. Outside Colab the form is a line such as `late_cost = 60`: edit the value and run the cell.

**Symbols.** The column «From» gives the version of the model where each symbol first appears. The warehouse is $w$.

| Symbol | Meaning | Unit | From |
|---|---|---|---|
| $F$, $f$ | Plants | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | M0 |
| $\tau$ | Last month of the horizon | — | M1 |
| $w$ | The central warehouse | — | M2 |
| $c_f$ | Unit assembly cost at plant $f$ | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | bikes/month | M0 |
| $d_t$ | Orders received in month $t$ | bikes/month | M0 |
| $c^h_f$; $c^h_w$ | Holding cost at plant $f$; at the warehouse | €/(bike·month) | M1; M2 |
| $h_{f0}$; $h_{w0}$ | Stock at plant $f$; at the warehouse, at the end of December | bikes | M1; M2 |
| $h^{\text{end}}_f$; $h^{\text{end}}_w$ | Stock that plant $f$; the warehouse, must hold at the end of the horizon | bikes | M1; M2 |
| $K^h_f$; $K^h_w$ | Storage capacity of plant $f$; of the warehouse | bikes | M2 |
| $c^b$ | Cost of one order served one month late | €/(bike·month) | M2 |
| $b_0$ | Orders owed at the end of December | bikes | M2 |
| $\bar b$ | Maximum orders owed at the end of the horizon | bikes | M2 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | bikes/month | M0 |
| $h_{ft}$; $h_{wt}$ | Bikes in stock at plant $f$; at the warehouse, at the end of month $t$ | bikes | M1; M2 |
| $x_{ft}$ | Bikes shipped from plant $f$ in month $t$, now to the warehouse | bikes/month | M1 |
| $s_t$ | Bikes the warehouse serves in month $t$, to orders of that month or of earlier months | bikes/month | M2 |
| $b_t$ | Orders received and not yet served at the end of month $t$ | bikes | M2 |

All the data and results are explained in the [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/central_warehouse.md). The data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import central_warehouse as cw

## 1. Plants, warehouse and orders

The first table gives, for each plant, the data of M1 and the one that M2 adds: how many bikes fit in its store. Holding a bike at a plant is cheaper than at the warehouse, but little fits there, because space at a plant is expensive. The second table gives the data of the central warehouse and of the orders served late: what serving one bike one month late costs, and how many orders may be owed at the end of December and at the end of June (none). The third table is forecast 3: the orders the warehouse receives each month, with a peak in February.

In [ ]:
# @title Data { display-mode: "form" }
cw.show_data()

## 2. The model with a central warehouse

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} \left( c_f\, p_{ft} + c^h_f\, h_{ft} \right) + c^h_w\, h_{wt} + c^b\, b_t \Big) \\
\text{s.t.} \quad & h_{f,t-1} + p_{ft} = x_{ft} + h_{ft} && \forall f \in F,\ t \in T && \text{(stock balance at each plant)} \\
& x_{ft} \le h_{f,t-1} + p_{ft} && \forall f \in F,\ t \in T && \text{(shipment limit at each plant)} \\
& h_{w,t-1} + \sum_{f \in F} x_{ft} = s_t + h_{wt} && \forall t \in T && \text{(stock balance at the warehouse)} \\
& s_t \le h_{w,t-1} + \sum_{f \in F} x_{ft} && \forall t \in T && \text{(serving limit at the warehouse)} \\
& b_{t-1} + d_t = s_t + b_t && \forall t \in T && \text{(balance of the orders)} \\
& p_{ft} \le K_{ft},\quad h_{ft} \le K^h_f && \forall f \in F,\ t \in T && \text{(capacity; storage at the plants)} \\
& h_{wt} \le K^h_w && \forall t \in T && \text{(storage at the warehouse)} \\
& b_{\tau} \le \bar b && && \text{(orders late at the end)} \\
& h_{f\tau} \ge h^{\text{end}}_f \quad \forall f \in F, \qquad h_{w\tau} \ge h^{\text{end}}_w && && \text{(final stock)} \\
& p_{ft},\ x_{ft},\ h_{ft},\ h_{wt},\ s_t,\ b_t \ge 0 && \forall f \in F,\ t \in T
\end{aligned}
$$

with $h_{f0}$, $h_{w0}$ and $b_0$ given. The plants ship to the warehouse. The warehouse has two balances. The stock: what it held at the start of the month plus what arrives is what it serves plus what it holds at the end. The orders: what it owed at the start of the month plus the orders of the month is what it serves plus what it still owes at the end. The bikes served, $s_t$, appear in both: each bike served leaves the stock and pays an order.

## 3. Plan with forecast 3

The cell first says why M1 has no plan for forecast 3. Then it solves M2 and gives, for each month, the bikes assembled at each plant, shipped to the warehouse and in stock at each plant, the stock of the warehouse, the bikes it serves and the orders it owes at the end of the month, with the cost of the plan. Then two tables of **shadow prices** (€/bike): what one more order each month would cost, and what one more bike of storage capacity at each site would save (negative: a saving). The figure on the left stacks the bikes assembled by each plant against the capacity of the plants, the orders and the bikes served; the one on the right stacks the stock at the plants and at the warehouse, with the orders owed beside it.

Where does the plan keep the bikes assembled in January, and in what order does it fill the stores? Why are some orders of February served in March, and not earlier? What would one more bike of storage at the warehouse in January be worth?

In [ ]:
# @title Plan with forecast 3 { display-mode: "form" }
cw.warehouse_plan();

## 4. Orders owed at the end of June

The plan may serve an order late, but no order may be owed at the end of June. The tick box removes that limit. The model does not see the months after June: for the model, an order that is never served costs only the months it waits until the end of June, 60 € each.

Run the cell without the limit. How many bikes does the plan assemble, and how many orders does it owe at the end of June? Why is that plan cheaper, and why would the company never accept it?

In [ ]:
# @title Orders owed at the end of June { display-mode: "form" }
limit_orders_at_the_end = True  # @param {type:"boolean"}
cw.orders_at_the_end_test(limit_orders_at_the_end);

## 5. Storage capacity at the plants

The slider sets how many bikes fit at plant 1. The capacities of plants 2 and 3 move with it, 10 and 20 bikes fewer, so that each plant can still hold the stock it had at the end of December. A plant is the cheapest place to keep a bike, but little fits there; what does not fit goes to the warehouse, which costs more and is also limited.

Give the plants more room, then less. Where does the stock assembled in January go, and how many orders of February are served late? Does a bike of storage at a plant save the same as one at the warehouse?

In [ ]:
# @title Storage capacity at the plants { display-mode: "form" }
storage_plant_1 = 40  # @param {type:"slider", min:20, max:200, step:10}
cw.plant_storage_test(storage_plant_1);

## 6. Cost of serving late, with forecast 2

This cell uses forecast 2 of M0 and M1, with its peak in April and May, and changes the cost of serving one bike one month late. Serving late is one more way to meet a peak, besides assembling in advance and using the dearest plant, and its cost decides whether the plan uses it.

Lower the cost from 60. Below what cost does the plan serve orders late instead of assembling them at plant 3, and which orders does it serve late? What happens at exactly that cost?

In [ ]:
# @title Cost of serving late { display-mode: "form" }
late_cost = 60  # @param {type:"slider", min:0, max:100, step:5}
cw.late_cost_test(late_cost);

## 7. Storage capacity of the warehouse

The slider sets how many bikes fit in the warehouse, with forecast 3. Bikes made in advance have to be kept somewhere until the orders arrive; a small warehouse limits how many can be made in advance.

Make the warehouse smaller, then larger. How many orders of February are served late in each case? From what capacity does more room change nothing, and why do some orders still have to be served late?

In [ ]:
# @title Storage capacity of the warehouse { display-mode: "form" }
warehouse_storage = 150  # @param {type:"slider", min:0, max:300, step:10}
cw.warehouse_storage_test(warehouse_storage);

All the numbers are explained in the [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/central_warehouse.md), which also has a script that reproduces them.